# 实践 9.3  模拟mGWAS与候选基因证据汇总

《统计遗传与智能育种学》第九章 · 电子资源 `09c_metabolite_mgwas.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：09c_metabolite_mgwas.ipynb。实践模拟单核苷酸多态性（SNP）、相关代谢特征、表达量和4个已知mQTL真值；先输出代谢特征相关矩阵，再逐代谢物进行单标记扫描，并把全部12×160个检验定义为同一检验族实施Benjamini–Hochberg校正。最后把显著位点按预设规则对应的基因和相应基因表达—代谢物相关证据合并成候选表。由于模拟真值已知，可以分别评价植入mQTL的检出情况与候选证据整合结果。

运行检查：重启内核并运行全部单元  
预期：输出代谢特征相关矩阵、全局BH校正后的mGWAS显著位点、植入真值找回数与非植入显著数，以及候选基因证据表；明确关联、预设基因对应和表达相关分别支持什么、不支持什么。

## 输入、完整代码、实际输出与解释

输入与可复现条件：12个相关代谢特征、160个SNP、表达量和4个预设mQTL真值；SEED=903；全部12×160检验定义为同一检验族进行全局BH校正。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED=903
rng=np.random.default_rng(SEED)
print('Python',sys.version.split()[0])
print('Platform',platform.platform())
print('numpy',np.__version__,'pandas',pd.__version__,'scikit-learn',sklearn.__version__)
print('SEED',SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 903


In [2]:
from scipy.stats import t as tdist
ns,ms,nmet=260,160,12; G=rng.binomial(2,rng.uniform(.1,.5,ms),(ns,ms)).astype(float); qtls=np.array([12,37,81,120]); M=rng.normal(0,1,(ns,nmet));
for k,q in enumerate(qtls): M[:,k]+=1.0*G[:,q]
latent=rng.normal(0,1,(ns,2)); M+=latent@rng.normal(0,.45,(2,nmet))
expr=rng.normal(0,1,(ns,32))
for q in qtls: expr[:,q//5]+=.7*G[:,q]
corr=np.corrcoef(M,rowvar=False); edges=np.argwhere(np.triu(np.abs(corr)>.35,1)); print('metabolite correlation-network edges |r|>0.35',len(edges)); print(np.round(corr,2))

def bh(p):
 p=np.asarray(p); order=np.argsort(p); ranked=p[order]*len(p)/np.arange(1,len(p)+1); q=np.minimum.accumulate(ranked[::-1])[::-1]; out=np.empty_like(q); out[order]=np.clip(q,0,1); return out
assoc=[]
for k in range(nmet):
 for j in range(ms):
  x=G[:,j]; X=np.column_stack([np.ones(ns),x]); b=np.linalg.lstsq(X,M[:,k],rcond=None)[0]; r=M[:,k]-X@b; se=np.sqrt((r@r)/(ns-2)*np.linalg.inv(X.T@X)[1,1]); p=2*tdist.sf(abs(b[1]/se),ns-2); assoc.append((k,j,p))
a=pd.DataFrame(assoc,columns=['metabolite','SNP','p']); a['q_global']=bh(a.p.to_numpy()); sig=a[a.q_global<.05]
planted={(k,int(q)) for k,q in enumerate(qtls)}; found=set(map(tuple,sig[['metabolite','SNP']].to_numpy())); tp=len(found & planted); fp=len(found-planted)
print('global-BH significant pairs',len(sig),'planted mQTL recovered',tp,'of',len(planted),'non-planted significant',fp)
# 候选证据：按模拟分组对应基因并列出表达相关，按关联校正P值排序
cand=sig.copy(); cand['gene_idx']=(cand.SNP//5).astype(int); cand['gene']=cand.gene_idx.map(lambda j:f'Gene{j:02d}')
cand['expr_support']=[abs(np.corrcoef(expr[:,gi],M[:,k])[0,1]) for gi,k in zip(cand.gene_idx,cand.metabolite)]
print(cand.nsmallest(10,'q_global')[['metabolite','SNP','gene','q_global','expr_support']].round(4).to_string(index=False))


metabolite correlation-network edges |r|>0.35 3
[[ 1.   -0.28 -0.35 -0.22  0.01 -0.    0.17 -0.22 -0.09 -0.09 -0.09  0.17]
 [-0.28  1.    0.43  0.29 -0.38  0.16 -0.11  0.28 -0.05  0.4  -0.29 -0.04]
 [-0.35  0.43  1.    0.26 -0.23  0.19 -0.05  0.26  0.01  0.32 -0.05 -0.04]
 [-0.22  0.29  0.26  1.   -0.16  0.09 -0.03  0.21  0.14  0.12  0.07 -0.09]
 [ 0.01 -0.38 -0.23 -0.16  1.   -0.14 -0.09 -0.2   0.07 -0.3   0.33 -0.08]
 [-0.    0.16  0.19  0.09 -0.14  1.    0.14  0.07 -0.14  0.15 -0.18 -0.04]
 [ 0.17 -0.11 -0.05 -0.03 -0.09  0.14  1.   -0.17 -0.18  0.06 -0.12  0.1 ]
 [-0.22  0.28  0.26  0.21 -0.2   0.07 -0.17  1.    0.01  0.07 -0.07 -0.04]
 [-0.09 -0.05  0.01  0.14  0.07 -0.14 -0.18  0.01  1.   -0.16  0.21 -0.02]
 [-0.09  0.4   0.32  0.12 -0.3   0.15  0.06  0.07 -0.16  1.   -0.29 -0.04]
 [-0.09 -0.29 -0.05  0.07  0.33 -0.18 -0.12 -0.07  0.21 -0.29  1.   -0.01]
 [ 0.17 -0.04 -0.04 -0.09 -0.08 -0.04  0.1  -0.04 -0.02 -0.04 -0.01  1.  ]]
global-BH significant pairs 7 planted mQTL recovere

结果解释：|r|>0.35的代谢相关网络有3条边；全局BH后得到7个显著代谢物—SNP对，4/4预设mQTL全部检出，同时有3个非植入显著结果。候选表把遗传关联、预设基因对应和表达相关并列为不同证据，任何一项都不能单独证明相应基因对该代谢物具有因果调控作用。

可修改练习：把BH从全局检验族改为按每个代谢物分别校正并比较发现数；说明两种检验族定义对应的错误发现率控制目标如何不同。